# Day 4: Prefill、Decode 和 KV Cache

选择 Python 3 (ipykernel)，从上到下运行。先用 Tiny 模型 FP32 证明机制，再检查本地真实 Qwen3。
7-token prefill + 1-token decode 的最后 logits 预测的是下一个 token，不是在输出输入的第 8 个 token。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "learning":
    root = root.parent
if not (root / "reasoning_from_scratch/qwen3.py").is_file():
    raise RuntimeError("请从仓库根目录或 learning 目录启动。")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.day03_shapes import TINY_CONFIG
from learning.day04_cache import compare_cache
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B, Qwen3Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Tiny FP32：严格验证 Cache

完整路径一次处理 8 token；Cache 路径先处理 7 个，再逐 token 处理最后 1 个。

In [ ]:
torch.manual_seed(123)
model = Qwen3Model(TINY_CONFIG).to(device).eval()
ids = torch.arange(8, device=device).unsqueeze(0)
report = compare_cache(model, ids, 7, atol=1e-5, rtol=1e-5)
print("Prefill K:", report["phases"][0]["cache"][0]["K"])
print("Decode 后 K:", report["phases"][-1]["cache"][0]["K"])
print("最大误差:", report["max_absolute_error"])
print("一致性:", report["logits_close"])

## 2. 五个问题

- Prefill 处理 prompt 的全部 token；这里是 7 个。
- Decode 只处理最新 token；历史 K/V 已在 Cache 中。
- Cache 保存 K/V，不保存历史 Q。
- GQA 让 16 个 Query heads 共享 8 组 K/V，每组服务 2 个 Query head。
- 完整前向和 Cache 路径的最后位置 logits 应在 dtype 容差内一致。

## 3. 真实 Qwen3（可选）

FP32 作为严格对照，BF16 记录最大误差和 top-1，不把 BF16 逐元素误差误判为逻辑错误。

In [ ]:
checkpoint = Path("qwen3/qwen3-0.6B-base.pth")
if device.type == "cuda" and checkpoint.is_file():
    for label, dtype in (("fp32", torch.float32), ("bf16", torch.bfloat16)):
        cfg = dict(QWEN_CONFIG_06_B, dtype=dtype)
        real = Qwen3Model(cfg).to(device).eval()
        real.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
        ids = torch.arange(8, device=device).unsqueeze(0)
        report = compare_cache(real, ids, 7, atol=1e-4, rtol=1e-4, strict=label == "fp32")
        print(label, "max error:", report["last_position_max_absolute_error"])
        print("close:", report["logits_close"], "top1:", report["last_top1_full"], report["last_top1_cached"])
        print("Prefill/Decode K:", report["phases"][0]["cache"][0]["K"], report["phases"][-1]["cache"][0]["K"])
else:
    print("需要 CUDA 和本地 qwen3/qwen3-0.6B-base.pth")

## 验收

- [ ] 能解释 Prefill、Decode 和 Cache 长度变化。
- [ ] 能写出 `[B, KV, T, D]` Cache 形状。
- [ ] 能解释 16 个 Query heads 共享 8 组 K/V。
- [ ] Tiny FP32 logits 一致性通过。
- [ ] 真实 Qwen3 能区分 FP32 严格误差和 BF16 观察结果。
